# 🤖 NLP Transformers: Common Problems & Solutions

This notebook covers the most common problems encountered when working with NLP Transformers and provides practical Python solutions for each.

---

## Table of Contents
1. [Vanishing/Exploding Gradients](#1)
2. [Attention Complexity (Quadratic Scaling)](#2)
3. [Positional Encoding Limitations](#3)
4. [Out-of-Vocabulary (OOV) Tokens](#4)
5. [Overfitting on Small Datasets](#5)
6. [Catastrophic Forgetting (Fine-Tuning)](#6)
7. [Long-Range Dependency Problems](#7)
8. [Memory & Computational Constraints](#8)
9. [Training Instability](#9)
10. [Hallucination & Factual Errors](#10)
11. [Tokenization Artifacts & Bias](#11)
12. [Class Imbalance in NLP Tasks](#12)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
<a id='1'></a>
## Problem 1: Vanishing / Exploding Gradients

### ❌ Problem
Deep transformer stacks suffer from gradients becoming too small (vanishing) or too large (exploding) during backpropagation, causing slow or unstable training.

### ✅ Solutions
- **Layer Normalization** (Pre-LN vs Post-LN)
- **Gradient Clipping**
- **Residual Connections**
- **Careful weight initialization**

In [ ]:
# ─── Solution 1a: Layer Normalization ───
class TransformerBlockWithLayerNorm(nn.Module):
    """Pre-LN Transformer block (more stable than Post-LN)"""
    def __init__(self, d_model=512, nhead=8, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)   # Pre-LN placement
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # Pre-LN: normalize BEFORE attention (more stable gradients)
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + self.drop(attn_out)   # residual connection
        x = x + self.drop(self.ff(self.norm2(x)))  # residual connection
        return x

block = TransformerBlockWithLayerNorm()
x = torch.randn(2, 10, 512)  # (batch, seq_len, d_model)
out = block(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Output mean: {out.mean().item():.4f}, std: {out.std().item():.4f}")

In [ ]:
# ─── Solution 1b: Gradient Clipping ───
model     = TransformerBlockWithLayerNorm()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

x      = torch.randn(2, 10, 512)
target = torch.randn(2, 10, 512)
loss   = nn.MSELoss()(model(x), target)

optimizer.zero_grad()
loss.backward()

# Compute gradient norm before clipping
total_norm_before = sum(p.grad.norm().item() ** 2
                        for p in model.parameters() if p.grad is not None) ** 0.5

# Clip gradients to max_norm=1.0  ← the key fix
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

total_norm_after = sum(p.grad.norm().item() ** 2
                       for p in model.parameters() if p.grad is not None) ** 0.5

print(f"Gradient norm BEFORE clipping: {total_norm_before:.4f}")
print(f"Gradient norm AFTER  clipping: {total_norm_after:.4f}")

optimizer.step()
print("✅ Training step completed safely.")

---
<a id='2'></a>
## Problem 2: Attention Complexity (Quadratic Scaling O(n²))

### ❌ Problem
Standard self-attention computes pairwise interactions between all tokens — memory and compute grow as O(n²) with sequence length, making very long sequences prohibitively expensive.

### ✅ Solutions
- **Sparse Attention** (only attend to nearby tokens)
- **Linear Attention** approximations
- **Sliding Window / Local Attention** (Longformer)
- **Flash Attention** (memory-efficient exact attention)

In [ ]:
# ─── Solution 2a: Visualise quadratic growth ───
seq_lengths = [128, 256, 512, 1024, 2048, 4096]
std_memory  = [n**2 for n in seq_lengths]
lin_memory  = [n    for n in seq_lengths]   # linear approx

plt.figure(figsize=(9, 4))
plt.plot(seq_lengths, std_memory, 'o-', label='Standard Attention O(n²)', color='tomato')
plt.plot(seq_lengths, lin_memory, 's--', label='Linear Attention O(n)',   color='steelblue')
plt.xlabel('Sequence Length'); plt.ylabel('Relative Memory Units')
plt.title('Attention Complexity: Standard vs Linear')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ─── Solution 2b: Sliding Window (Local) Attention ───
def sliding_window_attention(Q, K, V, window_size=4):
    """
    Each token only attends to 'window_size' neighbours.
    Reduces O(n²) → O(n * window_size).
    """
    B, T, D = Q.shape
    output  = torch.zeros_like(Q)
    scale   = D ** -0.5

    for i in range(T):
        lo  = max(0, i - window_size // 2)
        hi  = min(T, i + window_size // 2 + 1)

        k_win = K[:, lo:hi, :]           # (B, win, D)
        v_win = V[:, lo:hi, :]           # (B, win, D)
        q_i   = Q[:, i:i+1, :]          # (B, 1, D)

        scores  = (q_i @ k_win.transpose(-2, -1)) * scale  # (B, 1, win)
        weights = torch.softmax(scores, dim=-1)
        output[:, i, :] = (weights @ v_win).squeeze(1)

    return output

B, T, D = 2, 16, 64
Q = torch.randn(B, T, D)
K = torch.randn(B, T, D)
V = torch.randn(B, T, D)

out = sliding_window_attention(Q, K, V, window_size=4)
print(f"Input  shape: {Q.shape}")
print(f"Output shape: {out.shape}")
print("✅ Sliding window attention computed (each token sees only 4 neighbours)")

---
<a id='3'></a>
## Problem 3: Positional Encoding Limitations

### ❌ Problem
Transformers have no built-in sense of order. Standard sinusoidal or learned positional encodings:
- Don't generalise to lengths **longer than training**
- Can be sensitive to position distribution shift

### ✅ Solutions
- **Sinusoidal PE** (original, absolute)
- **Relative PE** (e.g. ALiBi, T5-style)
- **Rotary Position Embedding – RoPE** (used in LLaMA, Mistral)

In [ ]:
# ─── Solution 3a: Sinusoidal Positional Encoding ───
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             -(np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)   # even dims → sin
        pe[:, 1::2] = torch.cos(position * div_term)   # odd  dims → cos
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

# Visualise
d_model = 64
pe_layer = SinusoidalPositionalEncoding(d_model)
pe_vals  = pe_layer.pe[0, :50, :].detach().numpy()

plt.figure(figsize=(10, 4))
plt.imshow(pe_vals.T, aspect='auto', cmap='RdBu_r', origin='lower')
plt.colorbar(label='Encoding Value')
plt.xlabel('Position'); plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding (first 50 positions, 64 dims)')
plt.tight_layout(); plt.show()
print("✅ Sinusoidal PE visualised")

In [ ]:
# ─── Solution 3b: Rotary Position Embedding (RoPE) ───
def apply_rope(x, seq_dim=1):
    """
    Minimal RoPE implementation.
    Rotates pairs of dimensions by position-dependent angles.
    Generalises better to unseen lengths than absolute PE.
    """
    B, T, D = x.shape
    assert D % 2 == 0

    theta = 1.0 / (10000 ** (torch.arange(0, D, 2).float() / D))
    pos   = torch.arange(T).float().unsqueeze(1)  # (T, 1)
    angles = pos * theta.unsqueeze(0)              # (T, D/2)

    cos_ = torch.cos(angles).unsqueeze(0)  # (1, T, D/2)
    sin_ = torch.sin(angles).unsqueeze(0)  # (1, T, D/2)

    x1, x2 = x[..., ::2], x[..., 1::2]   # even, odd dims
    x_rot   = torch.stack([-x2, x1], dim=-1).flatten(-2)  # rotate

    cos_full = torch.cat([cos_, cos_], dim=-1).reshape(1, T, D)
    sin_full = torch.cat([sin_, sin_], dim=-1).reshape(1, T, D)

    return x * cos_full + x_rot * sin_full

x   = torch.randn(2, 16, 64)
out = apply_rope(x)
print(f"Input  shape: {x.shape}")
print(f"RoPE   shape: {out.shape}")
print("✅ RoPE applied — relative distances preserved in dot-product attention")

---
<a id='4'></a>
## Problem 4: Out-of-Vocabulary (OOV) Tokens

### ❌ Problem
Word-level tokenizers fail completely on unseen words (typos, neologisms, domain terms).

### ✅ Solutions
- **BPE (Byte-Pair Encoding)** — used in GPT models
- **WordPiece** — used in BERT
- **SentencePiece / Unigram** — language-agnostic
- **Byte-level tokenization** — truly zero OOV (GPT-2, LLaMA)

In [ ]:
# ─── Solution 4: Demonstrate Sub-word Tokenization (BERT WordPiece) ───
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

test_sentences = [
    "The transformer architecture revolutionized NLP.",
    "ChatGPT uses transformerized subword tokenization.",  # novel word
    "Supercalifragilisticexpialidocious is a long word.",
    "मैं हिंदी में लिख रहा हूँ।",  # non-English
]

for sent in test_sentences:
    tokens = tokenizer.tokenize(sent)
    ids    = tokenizer.encode(sent)
    print(f"Input   : {sent[:55]}..." if len(sent) > 55 else f"Input   : {sent}")
    print(f"Tokens  : {tokens}")
    print(f"# tokens: {len(tokens)}  | OOV: None (subword handles all!)")
    print()

---
<a id='5'></a>
## Problem 5: Overfitting on Small Datasets

### ❌ Problem
Transformers have millions of parameters and overfit easily when labelled data is scarce.

### ✅ Solutions
- **Dropout** regularisation
- **Weight decay** (L2 regularisation via AdamW)
- **Data augmentation** (back-translation, synonym replacement)
- **Pre-training + Fine-tuning** (transfer learning)

In [ ]:
# ─── Solution 5a: AdamW with weight decay & Dropout ───
class RegularisedTransformer(nn.Module):
    def __init__(self, vocab_size=30522, d_model=256, nhead=8,
                 num_layers=4, dropout=0.3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = SinusoidalPositionalEncoding(d_model, dropout=dropout)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model*4,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.encoder   = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout   = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask=None):
        x = self.pos_enc(self.embedding(input_ids))
        x = self.encoder(x, src_key_padding_mask=(attention_mask == 0) if attention_mask is not None else None)
        x = x.mean(dim=1)          # mean pooling
        return self.classifier(self.dropout(x))

model = RegularisedTransformer()

# AdamW decouples weight decay from gradient updates — critical for transformers
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01,   # L2 regularisation
    betas=(0.9, 0.999)
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters  : {total_params:,}")
print(f"Dropout rate      : 0.3")
print(f"Weight decay (L2) : 0.01")
print("✅ Regularised model + AdamW ready")

In [ ]:
# ─── Solution 5b: Simple text augmentation ───
import random

def synonym_swap(text, swap_prob=0.15):
    """Randomly swap words with simple synonyms for augmentation."""
    simple_synonyms = {
        'good': ['great', 'excellent', 'fine'],
        'bad':  ['poor', 'terrible', 'awful'],
        'fast': ['quick', 'rapid', 'swift'],
        'big':  ['large', 'huge', 'enormous'],
    }
    words = text.split()
    return ' '.join(
        random.choice(simple_synonyms[w]) if w in simple_synonyms and random.random() < swap_prob else w
        for w in words
    )

sentences = [
    "The model is good and fast for big datasets.",
    "This is a bad result from a big experiment.",
]

for s in sentences:
    augmented = [synonym_swap(s) for _ in range(3)]
    print(f"Original : {s}")
    for i, a in enumerate(augmented, 1):
        print(f"  Aug {i}  : {a}")
    print()

---
<a id='6'></a>
## Problem 6: Catastrophic Forgetting During Fine-Tuning

### ❌ Problem
When fine-tuning a pre-trained transformer on a specific task, it can forget general language knowledge.

### ✅ Solutions
- **Learning rate warmup + low LR**
- **Layer-wise LR decay** (lower LR for earlier layers)
- **LoRA / Adapter modules** (freeze most weights)
- **Gradual unfreezing**

In [ ]:
# ─── Solution 6a: Warmup + Cosine LR Schedule ───
from torch.optim.lr_scheduler import LambdaLR

def get_warmup_cosine_schedule(optimizer, warmup_steps, total_steps):
    """Linear warmup, then cosine decay — standard transformer schedule."""
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)

model     = RegularisedTransformer()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

warmup_steps = 100
total_steps  = 1000
scheduler    = get_warmup_cosine_schedule(optimizer, warmup_steps, total_steps)

lrs = []
for _ in range(total_steps):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(9, 3))
plt.plot(lrs, color='steelblue', lw=2)
plt.axvline(warmup_steps, color='tomato', linestyle='--', label=f'End of warmup ({warmup_steps} steps)')
plt.xlabel('Training Step'); plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay Schedule')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ─── Solution 6b: Minimal LoRA Implementation ───
class LoRALinear(nn.Module):
    """
    Low-Rank Adaptation: adds two small matrices A and B to a frozen weight.
    W' = W + alpha/r * (B @ A)   — only A and B are trained.
    """
    def __init__(self, in_features, out_features, r=4, alpha=16):
        super().__init__()
        self.r     = r
        self.alpha = alpha
        self.scaling = alpha / r

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.weight.requires_grad = False   # FROZEN

        # Trainable low-rank matrices
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        base   = nn.functional.linear(x, self.weight)
        delta  = nn.functional.linear(nn.functional.linear(x, self.lora_A), self.lora_B)
        return base + self.scaling * delta

layer        = LoRALinear(768, 768, r=8, alpha=16)
frozen_params   = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)

print(f"Frozen parameters   : {frozen_params:,}")
print(f"Trainable (LoRA)    : {trainable_params:,}")
print(f"Trainable ratio     : {trainable_params/(frozen_params+trainable_params)*100:.2f}%")
print("✅ LoRA: fine-tune with <1% of parameters, virtually no forgetting")

---
<a id='7'></a>
## Problem 7: Long-Range Dependency Failure

### ❌ Problem
Even though attention is global in theory, in practice transformers struggle to maintain coherence over thousands of tokens because attention weights diffuse.

### ✅ Solutions
- **Longer context windows** (GPT-4, Gemini)
- **Memory augmentation** (Memorizing Transformers)
- **Recurrent hybrids** (Mamba, RWKV)
- **Hierarchical chunking** at inference time

In [ ]:
# ─── Solution 7: Hierarchical Chunking for Long Documents ───
def hierarchical_encode(text: str, tokenizer, model,
                         chunk_size: int = 128, max_chunks: int = 8):
    """
    Encodes a long document by:
      1. Splitting into overlapping chunks
      2. Encoding each chunk independently
      3. Average-pooling chunk representations → single document vector
    """
    tokens   = tokenizer(text, return_tensors='pt', truncation=False)
    input_ids = tokens['input_ids'][0]
    total_len = input_ids.shape[0]

    chunk_embeddings = []
    stride = chunk_size // 2     # 50% overlap

    for start in range(0, min(total_len, chunk_size * max_chunks), stride):
        end   = min(start + chunk_size, total_len)
        chunk = input_ids[start:end].unsqueeze(0)
        with torch.no_grad():
            out = model(input_ids=chunk)
        # CLS token as chunk repr
        chunk_embeddings.append(out.last_hidden_state[:, 0, :])

    # Pool chunk embeddings → document embedding
    doc_embedding = torch.stack(chunk_embeddings, dim=1).mean(dim=1)
    return doc_embedding, len(chunk_embeddings)

from transformers import AutoModel, AutoTokenizer
tok   = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

long_text = ("The transformer model processes natural language. " * 60)  # ~480 tokens

doc_vec, n_chunks = hierarchical_encode(long_text, tok, model)
print(f"Document length  : {len(tok.tokenize(long_text))} tokens")
print(f"Chunks processed : {n_chunks}")
print(f"Doc embedding    : {doc_vec.shape}")
print("✅ Long document encoded via hierarchical chunking")

---
<a id='8'></a>
## Problem 8: Memory & Computational Constraints

### ❌ Problem
Large transformer models require significant GPU memory, limiting batch size and sequence length in practice.

### ✅ Solutions
- **Mixed-precision training** (FP16 / BF16)
- **Gradient accumulation** (simulate large batches)
- **Gradient checkpointing** (trade compute for memory)
- **Model quantisation** (INT8 / INT4)

In [ ]:
# ─── Solution 8a: Gradient Accumulation ───
def train_with_gradient_accumulation(
    model, optimizer, data_loader,
    accumulation_steps=4,   # effective_batch = batch_size * accumulation_steps
    num_steps=20,
):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer.zero_grad()

    for step, (input_ids, labels) in enumerate(data_loader):
        if step >= num_steps:
            break

        logits = model(input_ids)
        loss   = criterion(logits, labels) / accumulation_steps  # scale loss
        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            print(f"  Step {step+1:>3}: weights updated (effective batch #{(step+1)//accumulation_steps})")

# Fake dataloader
fake_data = [(torch.randint(0, 100, (4, 20)), torch.randint(0, 2, (4,))) for _ in range(20)]

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(100, 32)
        self.fc  = nn.Linear(32, 2)
    def forward(self, x):
        return self.fc(self.emb(x).mean(1))

tiny_model = TinyModel()
opt = torch.optim.Adam(tiny_model.parameters())
print("Gradient accumulation demo (4 mini-batches → 1 effective batch):")
train_with_gradient_accumulation(tiny_model, opt, fake_data)

In [ ]:
# ─── Solution 8b: Mixed-Precision Training (AMP) ───
from torch.cuda.amp import autocast, GradScaler

def train_amp_step(model, optimizer, scaler, x, y):
    """Single AMP training step — halves memory on GPU."""
    device = next(model.parameters()).device
    x, y   = x.to(device), y.to(device)

    optimizer.zero_grad()

    with autocast():                          # FP16 forward pass
        logits = model(x)
        loss   = nn.CrossEntropyLoss()(logits, y)

    scaler.scale(loss).backward()            # scaled backward
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)                   # safe weight update
    scaler.update()

    return loss.item()

scaler = GradScaler(enabled=torch.cuda.is_available())
x_demo = torch.randint(0, 100, (8, 20))
y_demo = torch.randint(0, 2, (8,))

loss_val = train_amp_step(tiny_model, opt, scaler, x_demo, y_demo)
print(f"AMP step loss: {loss_val:.4f}")
print(f"AMP enabled  : {torch.cuda.is_available()} (runs in FP32 fallback on CPU)")
print("✅ Mixed-precision training configured")

---
<a id='9'></a>
## Problem 9: Training Instability

### ❌ Problem
Transformers can exhibit sudden loss spikes, divergence, or NaN losses during training.

### ✅ Solutions
- **Careful initialisation** (Xavier / scaled init)
- **Loss monitoring + early stopping**
- **Learning rate warmup**
- **Label smoothing**

In [ ]:
# ─── Solution 9a: Xavier Initialisation ───
def init_transformer_weights(model):
    """Apply Xavier uniform init to linear layers and zeros to biases."""
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0, std=0.02)

model = RegularisedTransformer()

# Before init
w_before = model.classifier.weight.data.clone()
init_transformer_weights(model)
w_after  = model.classifier.weight.data

print(f"Weight std BEFORE init: {w_before.std().item():.4f}")
print(f"Weight std AFTER  init: {w_after.std().item():.4f}")
print("✅ Stable initialisation applied")

In [ ]:
# ─── Solution 9b: Label Smoothing ───
class LabelSmoothingLoss(nn.Module):
    """
    Prevents the model from becoming overconfident, 
    reducing instability from near-zero cross-entropy.
    """
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, logits, targets):
        log_probs = torch.log_softmax(logits, dim=-1)

        # Smoothed targets: true class gets (1 - ε), others get ε / (K-1)
        smooth_val = self.smoothing / (self.num_classes - 1)
        targets_one_hot = torch.full_like(log_probs, smooth_val)
        targets_one_hot.scatter_(-1, targets.unsqueeze(-1), 1.0 - self.smoothing)

        return -(targets_one_hot * log_probs).sum(dim=-1).mean()

criterion_standard = nn.CrossEntropyLoss()
criterion_smooth   = LabelSmoothingLoss(num_classes=10, smoothing=0.1)

logits  = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

loss_std    = criterion_standard(logits, targets)
loss_smooth = criterion_smooth(logits, targets)

print(f"Standard Cross-Entropy Loss : {loss_std.item():.4f}")
print(f"Label-Smoothed Loss (ε=0.1) : {loss_smooth.item():.4f}")
print("✅ Label smoothing reduces overconfidence and training instability")

---
<a id='10'></a>
## Problem 10: Hallucination & Factual Errors

### ❌ Problem
Transformers generate plausible-sounding but factually incorrect text because they model distributions, not facts.

### ✅ Solutions
- **Temperature calibration** (lower T = more factual)
- **Top-p / Top-k sampling** constraints
- **Retrieval-Augmented Generation (RAG)**
- **Uncertainty estimation** (Monte Carlo Dropout)

In [ ]:
# ─── Solution 10a: Temperature & Sampling Strategies ───
def sample_next_token(logits, temperature=1.0, top_k=0, top_p=1.0):
    """
    temperature: <1.0 → sharper (less hallucination), >1.0 → more random
    top_k      : keep only k highest-prob tokens
    top_p      : nucleus sampling — cumulative prob cutoff
    """
    logits = logits / max(temperature, 1e-9)

    if top_k > 0:
        vals, _ = torch.topk(logits, top_k)
        logits  = logits.masked_fill(logits < vals[-1], float('-inf'))

    probs = torch.softmax(logits, dim=-1)

    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        remove    = cum_probs - sorted_probs > top_p
        sorted_probs[remove] = 0
        probs = torch.zeros_like(probs).scatter_(0, sorted_idx, sorted_probs)
        probs = probs / probs.sum()

    return torch.multinomial(probs, 1).item()

# Compare distributions at different temperatures
vocab_size = 20
logits_ex  = torch.randn(vocab_size)
temps      = [0.3, 0.7, 1.0, 1.5]

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, t in zip(axes, temps):
    probs = torch.softmax(logits_ex / t, dim=-1).numpy()
    ax.bar(range(vocab_size), probs, color='steelblue', alpha=0.8)
    ax.set_title(f'T = {t}'); ax.set_xlabel('Token ID')
axes[0].set_ylabel('Probability')
plt.suptitle('Effect of Temperature on Token Distribution', fontsize=13)
plt.tight_layout(); plt.show()
print("✅ Lower T → peaked distribution → less hallucination")

In [ ]:
# ─── Solution 10b: Monte Carlo Dropout for Uncertainty ───
def mc_dropout_predict(model, x, n_samples=30):
    """Enable dropout at inference for uncertainty estimation."""
    model.train()   # keep dropout ON
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(x)
            preds.append(torch.softmax(logits, dim=-1))

    preds   = torch.stack(preds, dim=0)   # (n_samples, batch, classes)
    mean    = preds.mean(0)
    std_dev = preds.std(0)                # uncertainty!
    return mean, std_dev

x_in = torch.randint(0, 100, (3, 20))
mean_pred, uncertainty = mc_dropout_predict(tiny_model, x_in)

print("MC Dropout Uncertainty Estimation (30 forward passes):")
for i in range(3):
    print(f"  Sample {i+1}: pred={mean_pred[i].argmax().item()}, "
          f"confidence={mean_pred[i].max().item():.3f}, "
          f"uncertainty={uncertainty[i].max().item():.4f}")
print("✅ High std → uncertain prediction → potential hallucination risk")

---
<a id='11'></a>
## Problem 11: Tokenization Artifacts & Bias

### ❌ Problem
- Whitespace sensitivity causes inconsistent tokenisation
- Rare / low-resource languages are split into many subword pieces
- Models can behave differently on semantically identical inputs that differ only in spacing

### ✅ Solutions
- **Normalise inputs** before tokenisation
- **Byte-level tokenisers** (GPT-2 style)
- **SentencePiece** with unigram model
- **Audit tokeniser coverage** by language

In [ ]:
# ─── Solution 11: Tokenization Audit & Normalization ───
import unicodedata, re

def normalize_text(text: str) -> str:
    """Normalize unicode, collapse whitespace, remove control chars."""
    text = unicodedata.normalize('NFC', text)   # canonical unicode
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)  # remove control chars
    text = re.sub(r'\s+', ' ', text).strip()    # collapse whitespace
    return text

tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

inputs = [
    "Hello world",
    "Hello  world",         # double space → different tokens!
    "Hello\tworld",         # tab
    "Héllo wörld",          # accented characters
    "hello world",          # lowercase
]

print("=" * 60)
print(f"{'Input':<30} {'Raw #tok':>8} {'Norm #tok':>10}")
print("=" * 60)
for s in inputs:
    raw_tok  = tok_bert.tokenize(s)
    norm_tok = tok_bert.tokenize(normalize_text(s))
    print(f"{repr(s):<30} {len(raw_tok):>8} {len(norm_tok):>10}")
print()
print("✅ Normalise inputs to avoid tokenisation artifacts")

---
<a id='12'></a>
## Problem 12: Class Imbalance in NLP Classification

### ❌ Problem
Real-world datasets are rarely balanced (e.g., 95% negative, 5% positive). Transformers will learn to predict the majority class.

### ✅ Solutions
- **Class-weighted loss**
- **Focal loss** (down-weight easy examples)
- **Oversampling / undersampling**
- **Threshold tuning**

In [ ]:
# ─── Solution 12a: Class-Weighted Cross-Entropy ───
# Simulate imbalanced dataset: 90% class 0, 10% class 1
n_total  = 1000
labels   = torch.zeros(n_total, dtype=torch.long)
labels[int(0.9 * n_total):] = 1

class_counts  = torch.bincount(labels).float()
class_weights = 1.0 / class_counts            # inverse frequency weighting
class_weights = class_weights / class_weights.sum()

print("Class distribution:")
for i, (cnt, wt) in enumerate(zip(class_counts.int().tolist(), class_weights.tolist())):
    print(f"  Class {i}: {cnt:>5} samples  →  weight: {wt:.4f}")

criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

logits  = torch.randn(16, 2)
targets = torch.randint(0, 2, (16,))

print(f"\nStandard CE loss : {nn.CrossEntropyLoss()(logits, targets).item():.4f}")
print(f"Weighted CE loss : {criterion_weighted(logits, targets).item():.4f}")

In [ ]:
# ─── Solution 12b: Focal Loss ───
class FocalLoss(nn.Module):
    """
    Focal Loss = -(1 - p_t)^gamma * log(p_t)
    gamma > 0 down-weights easy examples, forcing the model
    to focus on hard minority-class samples.
    """
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(logits, targets)
        pt      = torch.exp(-ce_loss)                        # probability of correct class
        return ((1 - pt) ** self.gamma * ce_loss).mean()

focal = FocalLoss(gamma=2.0, alpha=class_weights)
ce    = nn.CrossEntropyLoss()(logits, targets)
fl    = focal(logits, targets)

print(f"Cross-Entropy Loss : {ce.item():.4f}")
print(f"Focal Loss (γ=2)   : {fl.item():.4f}")
print("\n✅ Focal Loss reduces the loss contribution from easy/majority-class examples")

---

## 📋 Summary Table

| # | Problem | Key Solution(s) |
|---|---------|----------------|
| 1 | Vanishing/Exploding Gradients | Pre-LN, Gradient Clipping, Residual Connections |
| 2 | Quadratic Attention Complexity | Sparse / Sliding Window Attention, Flash Attention |
| 3 | Positional Encoding Limits | RoPE, ALiBi, Relative PE |
| 4 | Out-of-Vocabulary Tokens | BPE, WordPiece, Byte-level tokenisation |
| 5 | Overfitting | AdamW + Weight Decay, Dropout, Data Augmentation |
| 6 | Catastrophic Forgetting | LR Warmup, LoRA, Gradual Unfreezing |
| 7 | Long-Range Dependencies | Hierarchical Chunking, Extended Context, RAG |
| 8 | Memory Constraints | AMP, Gradient Accumulation, Quantisation |
| 9 | Training Instability | Xavier Init, Label Smoothing, LR Schedule |
| 10 | Hallucination | Temperature Tuning, MC Dropout, RAG |
| 11 | Tokenisation Artifacts | Input Normalisation, Byte-level Tokeniser |
| 12 | Class Imbalance | Weighted CE, Focal Loss, Threshold Tuning |

---

> **Note:** Each solution above is self-contained and can be adapted to your specific model and task. Start with the simplest fix (e.g., gradient clipping, weight decay) before adopting more complex approaches like LoRA or Flash Attention.